# MEGAN 3.2 Biogenic Emissions Notebook (WRF-driven, Santa Catarina)

This notebook implements a **MEGAN 3.2-oriented workflow** for Santa Catarina, Brazil, using the
same practical gamma-factor framework widely used in MEGAN2.1 regional applications
(Guenther et al., 2006; Guenther et al., 2012) and compatible with BRAIN-style WRF-driven inputs.

Core equation implemented in the code cells:

\[
E = EF_v 	imes LAI 	imes \gamma_{LAI} 	imes \left(LDF	imes\gamma_{T,light}	imes\gamma_P + (1-LDF)	imes\gamma_{TI}ight)
\]

with standard regional assumptions \(\gamma_{SM}=1\) and \(ho=1\).

Outputs are CF-1.8 NetCDF with EPSG:4326 metadata and `latitude`/`longitude` curvilinear coordinates,
plus quick-look maps for QA/QC.


## 1. User Configuration

**Edit the cell below** to set your file paths, simulation period, and meteorological source.
All paths are relative to this notebook's location (`notebook/`).


## 1A. Accessibility & Configuration Decisions (Requested Questions)

1. **CodeOcean capsule access**: the notebook now supports `METEOROLOGICAL_SOURCE='satellite'` and first attempts ERA5 retrieval from **CodeOcean capsule 4836770** (when credentials/endpoints are provided).
2. **CDS fallback**: if the CodeOcean attempt is unavailable, the same satellite workflow automatically falls back to **Copernicus CDS** (`cdsapi`) and caches downloaded files.
3. **SciDB WRF integration**: `METEOROLOGICAL_SOURCE='wrf'` retains both **local files** and **link-list download** (`wrf_2019_list.txt`) modes.
4. **Small-memory recommendation**: keep WRF as local one-file-at-a-time processing; satellite mode also caches daily NetCDF files to avoid repeated downloads.
5. **EPSG verification**: output NetCDF includes CF `crs` variable (`grid_mapping_name=latitude_longitude`) with WGS84/EPSG:4326 attributes and explicit `latitude`/`longitude` 2-D coordinates.


In [ ]:
# ===========================================================================
#  USER CONFIGURATION — EDIT THIS CELL
# ===========================================================================
import os
from datetime import datetime

# --- Time period (inclusive) ---
START_DATE = datetime(2020, 1, 1)
END_DATE   = datetime(2020, 1, 31)

# --- Base directory (works from VS Code + Colab + either notebook path) ---
cwd = os.getcwd()
if os.path.basename(cwd).lower() == 'notebook':
    BASE_DIR = os.path.abspath(os.path.join(cwd, '..'))
else:
    BASE_DIR = cwd

# --- Input/output paths ---
WRF_DIR    = os.path.join(BASE_DIR, 'input', 'WRF')
LAI_FILE   = os.path.join(BASE_DIR, 'input', 'LAI', 'MEGAN_LAI_CLIM_d02_MEGAN32.nc')
CT3_FILE   = os.path.join(BASE_DIR, 'input', 'LULC', 'CT3.SC_Brazil.csv')
EF_DIR     = os.path.join(BASE_DIR, 'input', 'EF')
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')
INTERMEDIATE_DIR = os.path.join(BASE_DIR, 'intermediate')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

# --- Meteorological source selection ---
# 'wrf'       -> current WRF pathway (local files or SciDB link list)
# 'satellite' -> ERA5 pathway (CodeOcean capsule 4836770 first, CDS fallback)
METEOROLOGICAL_SOURCE = 'wrf'

# --- WRF source mode (used only when METEOROLOGICAL_SOURCE='wrf') ---
# 'local' -> read existing files in input/WRF/
# 'links' -> read URLs from wrf_2019_list.txt, download/extract on-demand
WRF_SOURCE_MODE = 'local'
WRF_LINK_LIST_FILE = os.path.join(WRF_DIR, 'wrf_2019_list.txt')
WRF_CACHE_DIR = os.path.join(INTERMEDIATE_DIR, 'wrf_cache')

# --- Satellite/ERA5 options (used only when METEOROLOGICAL_SOURCE='satellite') ---
SATELLITE_CACHE_DIR = os.path.join(INTERMEDIATE_DIR, 'satellite_cache')
# Bounding box tuple: (south, west, north, east). If None, infer from first WRF file grid.
SATELLITE_BBOX = None

# --- WRF file naming pattern ---
WRF_PREFIX = 'wrfout_d02_'

# --- Model controls ---
N_CANOPY_LAYERS = 5
WARMUP_HOURS    = 240

# --- Core MEGAN species solved internally ---
SPECIES_LIST = [
    'ISOP', 'MYRC', 'SABI', 'LIMO', 'A_3CAR', 'OCIM', 'BPIN', 'APIN',
    'OMTP', 'FARN', 'BCAR', 'OSQT', 'MBO', 'MEOH', 'ACTO', 'CO',
    'NO', 'BIDER', 'STRESS', 'OTHER'
]

# --- Reporting species requested for outputs ---
REPORT_SPECIES = ['ISOP', 'MTRY', 'SESQ', 'CH3OH', 'HCHO', 'CH3COOH', 'OTHER_VOC']

print('Configuration loaded')
print(f'  BASE_DIR: {BASE_DIR}')
print(f'  Date range: {START_DATE:%Y-%m-%d} → {END_DATE:%Y-%m-%d}')
print(f'  Meteorological source: {METEOROLOGICAL_SOURCE}')
if METEOROLOGICAL_SOURCE == 'wrf':
    print(f'  WRF mode: {WRF_SOURCE_MODE}')
else:
    print(f'  Satellite cache: {SATELLITE_CACHE_DIR}')


In [ ]:
# ===========================================================================
# OPTIONAL: Colab bootstrap (runs only on Colab)
# ===========================================================================
import sys

if 'google.colab' in sys.modules:
    !pip -q install numpy==1.26.4 pandas==2.2.2 xarray==2024.3.0 netCDF4==1.6.5 scipy==1.12.0 matplotlib==3.8.4 requests==2.32.3 cdsapi==0.7.7

# Optional CDS setup (Colab): provide CDSAPI_KEY as "<uid>:<api_key>"
cds_key = os.environ.get('CDSAPI_KEY')
if cds_key and not os.path.exists(os.path.expanduser('~/.cdsapirc')):
    with open(os.path.expanduser('~/.cdsapirc'), 'w', encoding='utf-8') as f:
        f.write('url: https://cds.climate.copernicus.eu/api\n')
        f.write(f'key: {cds_key}\n')

for p in [WRF_DIR, os.path.dirname(LAI_FILE), os.path.dirname(CT3_FILE), EF_DIR, OUTPUT_DIR, INTERMEDIATE_DIR, SATELLITE_CACHE_DIR]:
    os.makedirs(p, exist_ok=True)

print('Environment/bootstrap checks complete.')


## 2. Imports and Dependencies

Required packages: `numpy`, `xarray`, `pandas`, `netCDF4`, `scipy`.
Install with: `pip install numpy xarray pandas netCDF4 scipy`


In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from datetime import datetime, timedelta
import glob
import math
import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

print("All imports successful.")


## 3. MEGAN Parameters

All parameter tables below are ported **verbatim** from GEE-MEGAN's
`Echo_parameter.py` ([Zenodo source](https://doi.org/10.5281/zenodo.15714886)).

These encode:
- **Compound emission factors** (EF, µg m⁻² h⁻¹) for 20 BVOC species × 16 PFTs
- **Leaf-age relative emission activities** (A_new, A_gro, A_mat, A_old)
- **Temperature-dependence parameters** (CLeo, Ct_m1)
- **Light-dependence fractions** (LDF)
- **Temperature-dependence factors** (β for light-independent pathway)


In [ ]:
# =========================================================================
# MEGAN2.1 Parameter Tables  (from GEE-MEGAN Echo_parameter.py)
# =========================================================================

# --- Species list (canonical order) ---
MGN_SPC = [
    'ISOP', 'MYRC', 'SABI', 'LIMO', 'A_3CAR', 'OCIM', 'BPIN', 'APIN',
    'OMTP', 'FARN', 'BCAR', 'OSQT', 'MBO', 'MEOH', 'ACTO', 'CO',
    'NO', 'BIDER', 'STRESS', 'OTHER'
]

# --- Leaf age → species category mapping (REA_SPC) ---
# Index into REL_EM_ACT arrays
REA_SPC = {
    'ISOP': 4, 'MYRC': 1, 'SABI': 1, 'LIMO': 1, 'A_3CAR': 1,
    'OCIM': 1, 'BPIN': 1, 'APIN': 1, 'OMTP': 1, 'FARN': 2,
    'BCAR': 2, 'OSQT': 2, 'MBO': 4, 'MEOH': 3, 'ACTO': 0,
    'CO': 0, 'NO': 0, 'BIDER': 0, 'STRESS': 0, 'OTHER': 0
}

# --- Relative emission activity by leaf developmental stage ---
# 5 categories; index from REA_SPC
REL_EM_ACT = {
    'Anew': [1.0, 2.0, 0.4, 3.5, 0.05],
    'Agro': [1.0, 1.8, 0.6, 3.0, 0.6],
    'Amat': [1.0, 1.0, 1.0, 1.0, 1.0],
    'Aold': [1.0, 1.05, 0.95, 1.2, 0.9],
}

# --- Canopy temperature-response constants per species ---
# CLeo: Eopt scaling; Ctm1: activation energy (Ea1t99 formula)
CLEO = {
    'ISOP': 2.0, 'MYRC': 1.83, 'SABI': 1.83, 'LIMO': 1.83,
    'A_3CAR': 1.83, 'OCIM': 1.83, 'BPIN': 1.83, 'APIN': 1.83,
    'OMTP': 1.83, 'FARN': 2.37, 'BCAR': 2.37, 'OSQT': 2.37,
    'MBO': 2.0, 'MEOH': 1.6, 'ACTO': 1.83, 'CO': 1.6,
    'NO': 1.83, 'BIDER': 2.0, 'STRESS': 1.83, 'OTHER': 1.83
}
CTM1 = {
    'ISOP': 95.0, 'MYRC': 80.0, 'SABI': 80.0, 'LIMO': 80.0,
    'A_3CAR': 80.0, 'OCIM': 80.0, 'BPIN': 80.0, 'APIN': 80.0,
    'OMTP': 80.0, 'FARN': 130.0, 'BCAR': 130.0, 'OSQT': 130.0,
    'MBO': 95.0, 'MEOH': 60.0, 'ACTO': 80.0, 'CO': 60.0,
    'NO': 80.0, 'BIDER': 95.0, 'STRESS': 80.0, 'OTHER': 80.0
}

# --- Light-Dependence Fraction (LDF) per species ---
LDF = {
    'ISOP': 0.999, 'MYRC': 0.6, 'SABI': 0.6, 'LIMO': 0.4,
    'A_3CAR': 0.4, 'OCIM': 0.4, 'BPIN': 0.4, 'APIN': 0.6,
    'OMTP': 0.4, 'FARN': 0.5, 'BCAR': 0.5, 'OSQT': 0.5,
    'MBO': 0.999, 'MEOH': 0.8, 'ACTO': 0.2, 'CO': 0.999,
    'NO': 0.0, 'BIDER': 0.8, 'STRESS': 0.8, 'OTHER': 0.2
}

# --- Temperature-dependence parameter for light-INDEPENDENT pathway ---
# β in: γ_TI = exp(β × (T - 303.15))   [Ealti99]
TDF_PRM = {
    'ISOP': 0.13, 'MYRC': 0.1, 'SABI': 0.1, 'LIMO': 0.1,
    'A_3CAR': 0.1, 'OCIM': 0.1, 'BPIN': 0.1, 'APIN': 0.1,
    'OMTP': 0.1, 'FARN': 0.17, 'BCAR': 0.17, 'OSQT': 0.17,
    'MBO': 0.13, 'MEOH': 0.08, 'ACTO': 0.1, 'CO': 0.08,
    'NO': 0.1, 'BIDER': 0.13, 'STRESS': 0.1, 'OTHER': 0.1
}

# --- Compound Emission Factors (µg m⁻² h⁻¹) per PFT ---
# 16 PFTs: indices 0–15 map to MEGAN2.1 standard PFT categories
# (see Guenther et al. 2012, Table 2)
#  0: NT_EG_Temp   1: NT_EG_Bor    2: NT_DC_Bor    3: BT_EG_Trop
#  4: BT_EG_Temp   5: BT_DC_Trop   6: BT_DC_Temp   7: BT_DC_Bor
#  8: SB_EG_Temp   9: SB_DC_Temp  10: SB_DC_Bor   11: GS_C3_Arct
# 12: GS_C3_NArc  13: GS_C4       14: Crop        15: Other/bare
COMPOUND_EF = {
    'ISOP':   [600, 3000, 1, 7000, 10000, 7000, 10000, 11000, 2000, 4000, 4000, 1600, 800, 200, 1, 0],
    'MYRC':   [70, 70, 60, 80, 30, 80, 30, 30, 30, 50, 30, 0.3, 0.3, 0.3, 0.3, 0],
    'SABI':   [70, 70, 40, 80, 50, 80, 50, 50, 50, 70, 50, 0.7, 0.7, 0.7, 0.7, 0],
    'LIMO':   [100, 100, 130, 80, 80, 80, 80, 80, 60, 100, 60, 0.7, 0.7, 0.7, 0.7, 0],
    'A_3CAR': [160, 160, 80, 40, 30, 40, 30, 30, 30, 100, 30, 0.3, 0.3, 0.3, 0.3, 0],
    'OCIM':   [70, 70, 60, 150, 120, 150, 120, 120, 90, 150, 90, 2, 2, 2, 2, 0],
    'BPIN':   [300, 300, 200, 120, 130, 120, 130, 130, 100, 150, 100, 1.5, 1.5, 1.5, 1.5, 0],
    'APIN':   [500, 500, 510, 600, 400, 600, 400, 400, 200, 300, 200, 2, 2, 2, 2, 0],
    'OMTP':   [180, 180, 170, 150, 150, 150, 150, 150, 110, 200, 110, 5, 5, 5, 5, 0],
    'FARN':   [40, 40, 40, 60, 40, 60, 40, 40, 40, 40, 40, 3, 3, 3, 4, 0],
    'BCAR':   [80, 80, 80, 60, 40, 60, 40, 40, 50, 50, 50, 1, 1, 1, 4, 0],
    'OSQT':   [120, 120, 120, 120, 100, 120, 100, 100, 100, 100, 100, 2, 2, 2, 2, 0],
    'MBO':    [700, 60, 0.01, 0.01, 0.01, 0.01, 0.01, 2, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0],
    'MEOH':   [900, 900, 900, 500, 900, 500, 900, 900, 900, 900, 900, 500, 500, 500, 900, 0],
    'ACTO':   [240, 240, 240, 240, 240, 240, 240, 240, 240, 240, 240, 80, 80, 80, 80, 0],
    'CO':     [600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 600, 0],
    'NO':     [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 27, 27, 27, 40, 68, 0],
    'BIDER':  [500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 500, 80, 80, 80, 80, 0],
    'STRESS': [300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 300, 0],
    'OTHER':  [140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 140, 0],
}

# --- Growth-form → PFT index mapping for Southern Brazil ---
# Used to select EF values from the 16-PFT table above.
# Customizable: change indices to match your domain's ecology.
GROWTHFORM_PFT_INDEX = {
    'NEEDL': 0,   # Needleleaf evergreen temperate tree
    'TROPI': 3,   # Broadleaf evergreen tropical tree
    'BROAD': 5,   # Broadleaf deciduous tropical tree (SC Brazil subtropical)
    'SHRUB': 9,   # Shrub broadleaf deciduous temperate
    'HERB':  13,  # C4 grass (dominant in SC Brazil)
    'CROP':  14,  # Cropland
}

# --- Canopy radiation parameters ---
CCE = 0.56  # canopy environment coefficient (Guenther et al., 2006)

print("MEGAN parameters loaded: 20 species × 16 PFTs")
print(f"Growth-form → PFT mapping: {GROWTHFORM_PFT_INDEX}")


## 4. MEGAN Gamma (γ) Activity Factor Functions

These functions implement the MEGAN2.1 activity factors that modulate
standard emission factors based on environmental conditions.

Each function is ported from `gamma_lib.py` and `canopy_lib.py` in the
GEE-MEGAN code, replacing `ee.Image` operations with NumPy array operations.

### Governing equations

| Factor | Description | Reference |
|--------|-------------|-----------|
| $\gamma_{LAI}$ | LAI scaling | Guenther 2006, Eq. 4 |
| $\gamma_T$ (light-dep.) | Temperature activity (Ea1t99) | Guenther 2012, Eq. 10 |
| $\gamma_P$ (light-dep.) | PPFD activity (Ea1p99) | Guenther 2012, Eq. 11 |
| $\gamma_{TI}$ (light-indep.) | Temperature (exponential, Ealti99) | Guenther 2012, Eq. 12 |
| $\gamma_{age}$ | Leaf age | Guenther 2006, Eq. 16 |
| $\gamma_{SM}$ | Soil moisture | (set to 1.0 in this version) |


In [ ]:
# =========================================================================
# MEGAN GAMMA FUNCTIONS  (NumPy / array-based)
# =========================================================================

def gamma_lai(LAI):
    """
    LAI scaling factor  (Guenther et al. 2006, Eq. 4).
    
    γ_LAI = 0.49 × LAI / sqrt(1 + 0.2 × LAI²)
    
    Parameters
    ----------
    LAI : np.ndarray
        Leaf Area Index (m² m⁻²), shape (ny, nx).
    
    Returns
    -------
    np.ndarray  : γ_LAI, same shape.
    """
    return 0.49 * LAI / np.sqrt(1.0 + 0.2 * LAI**2)


def gamma_T_lightdep(T, T24, T240, spc_name):
    """
    Temperature activity factor for LIGHT-DEPENDENT emissions (Ea1t99).
    
    Guenther et al. (2012), Eq. 10:
        T_opt = 312.5 + 0.6 × (T240 − 297)
        x     = (1/T_opt − 1/T) / 0.00831
        E_opt = C_Leo × exp(0.05×(T24−297)) × exp(0.05×(T240−297))
        γ_T   = E_opt × C_T2 × exp(x × C_T1) / [C_T2 − C_T1 × (1 − exp(x × C_T2))]
    
    where C_T2 = 230 (universal); C_Leo, C_T1 are species-specific.
    
    Parameters
    ----------
    T    : np.ndarray  – Instantaneous leaf/air temperature (K)
    T24  : np.ndarray  – 24-hour mean temperature (K)
    T240 : np.ndarray  – 240-hour (10-day) mean temperature (K)
    spc_name : str     – Species code (e.g. 'ISOP')
    
    Returns
    -------
    np.ndarray : γ_T (light-dependent), same shape as T.
    """
    Ct2  = 230.0
    Cleo = CLEO[spc_name]
    Ct1  = CTM1[spc_name]
    
    Topt = 312.5 + 0.6 * (T240 - 297.0)
    x    = (1.0 / Topt - 1.0 / T) / 0.00831
    
    Eopt = Cleo * np.exp(0.05 * (T24 - 297.0)) * np.exp(0.05 * (T240 - 297.0))
    
    numerator   = Ct2 * np.exp(x * Ct1)
    denominator = Ct2 - Ct1 * (1.0 - np.exp(x * Ct2))
    
    gamma = Eopt * numerator / denominator
    
    # Below 260 K: no emission
    gamma = np.where(T >= 260.0, gamma, 0.0)
    
    # Clamp to avoid numerical explosion
    gamma = np.clip(gamma, 0.0, 100.0)
    
    return gamma


def gamma_P_lightdep(PPFD, PPFD24, PPFD240, Pstd=200.0):
    """
    Light activity factor for LIGHT-DEPENDENT emissions (Ea1p99).
    
    Guenther et al. (2012), Eq. 11:
        α     = 0.004 − 0.0005 × ln(PPFD240)
        C1    = 0.0468 × exp(0.0005 × (PPFD24 − Pstd)) × PPFD240^0.6
        γ_P   = α × C1 × PPFD / sqrt(α² × PPFD² + 1)
    
    Parameters
    ----------
    PPFD    : np.ndarray – Instantaneous PPFD (µmol m⁻² s⁻¹)
    PPFD24  : np.ndarray – 24-hour mean PPFD
    PPFD240 : np.ndarray – 240-hour mean PPFD
    Pstd    : float      – Standard PPFD (200 for sunlit, 50 for shade)
    
    Returns
    -------
    np.ndarray : γ_P (light-dependent), same shape.
    """
    # Guard against log(0)
    PPFD240_safe = np.maximum(PPFD240, 0.01)
    
    alpha = 0.004 - 0.0005 * np.log(PPFD240_safe)
    C1    = 0.0468 * np.exp(0.0005 * (PPFD24 - Pstd)) * PPFD240_safe**0.6
    
    gamma = alpha * C1 * PPFD / np.sqrt(alpha**2 * PPFD**2 + 1.0)
    gamma = np.where(PPFD240 >= 0.01, gamma, 0.0)
    gamma = np.clip(gamma, 0.0, 100.0)
    
    return gamma


def gamma_T_lightindep(T, spc_name):
    """
    Temperature factor for LIGHT-INDEPENDENT emissions (Ealti99).
    
    Guenther et al. (2012), Eq. 12:
        γ_TI = exp(β × (T − 303.15))
    
    where β is species-specific (TDF_PRM).
    
    Parameters
    ----------
    T        : np.ndarray – Temperature (K)
    spc_name : str
    
    Returns
    -------
    np.ndarray : γ_TI
    """
    beta = TDF_PRM[spc_name]
    return np.exp(beta * (T - 303.15))


def gamma_age(LAI_current, LAI_previous, T_daily_mean, TSTLEN=30):
    """
    Leaf age factor (GAMMA_A from gamma_lib.py).
    
    Computes fractional age distribution (F_new, F_gro, F_mat, F_old)
    based on LAI change between current and previous month, then weights
    by species-specific relative emission activities.
    
    Parameters
    ----------
    LAI_current  : np.ndarray – Current month LAI
    LAI_previous : np.ndarray – Previous month LAI
    T_daily_mean : np.ndarray – Daily mean temperature (K)
    TSTLEN       : int        – Time step length (days between LAI obs)
    
    Returns
    -------
    dict : {spc_name: np.ndarray of γ_age} for all 20 species.
    """
    t  = TSTLEN
    Tt = T_daily_mean
    
    # ti: number of days for new leaves to mature
    ti = np.where(Tt <= 303.0, 5.0 + (-0.7) * (Tt - 300.0), 2.9)
    tm = 2.3 * ti  # maturation time
    
    # Fractional age distributions
    lai_c = LAI_current
    lai_p = LAI_previous
    
    # Avoid division by zero
    lai_c_safe = np.maximum(lai_c, 1e-6)
    lai_p_safe = np.maximum(lai_p, 1e-6)
    ratio = lai_p / lai_c_safe
    
    # Growing canopy (lai_p < lai_c)
    growing = lai_p < lai_c
    # Fnew for growing
    Fnew_grow = np.where(ti >= t, 1.0 - ratio, (ti / t) * (1.0 - ratio))
    # Fmat for growing
    Fmat_grow = np.where(tm >= t, ratio, ratio + (t - tm) / t * (1.0 - ratio))
    # Fgro for growing
    Fgro_grow = np.maximum(1.0 - Fnew_grow - Fmat_grow, 0.0)
    # Fold for growing
    Fold_grow = np.zeros_like(lai_c)
    
    # Stable canopy (lai_p ≈ lai_c)
    stable = np.isclose(lai_p, lai_c, rtol=1e-4) | (lai_p == lai_c)
    
    # Declining canopy (lai_p > lai_c)
    declining = lai_p > lai_c
    diff_ratio = np.where(lai_p_safe > 0, (lai_p - lai_c) / lai_p_safe, 0.0)
    
    # Assemble fractions
    Fnew = np.where(growing, Fnew_grow, np.where(stable, 0.0, 0.0))
    Fgro = np.where(growing, Fgro_grow, np.where(stable, 0.1, 0.0))
    Fmat = np.where(growing, Fmat_grow, np.where(stable, 0.8, 1.0 - diff_ratio))
    Fold = np.where(growing, Fold_grow, np.where(stable, 0.1, diff_ratio))
    
    # Clamp fractions to [0, 1]
    Fnew = np.clip(Fnew, 0, 1)
    Fgro = np.clip(Fgro, 0, 1)
    Fmat = np.clip(Fmat, 0, 1)
    Fold = np.clip(Fold, 0, 1)
    
    # Compute species-specific gamma_age
    result = {}
    for spc in MGN_SPC:
        cat = REA_SPC[spc]
        Anew = REL_EM_ACT['Anew'][cat]
        Agro = REL_EM_ACT['Agro'][cat]
        Amat = REL_EM_ACT['Amat'][cat]
        Aold = REL_EM_ACT['Aold'][cat]
        gam_a = Fnew * Anew + Fgro * Agro + Fmat * Amat + Fold * Aold
        result[spc] = gam_a
    
    return result


print("Gamma functions defined: γ_LAI, γ_T (LD), γ_P (LD), γ_TI (LI), γ_age")


## 5. Solar Geometry and Radiation Partitioning

These functions partition incoming shortwave radiation (from WRF `SWDOWN`) into
**beam** and **diffuse** components for **visible (PAR)** and **near-infrared (NIR)**
wavelengths, following the algorithms in GEE-MEGAN's `canopy_lib.py`.

The sunlit fraction of the canopy at each depth is computed using Beer's law
extinction.


In [ ]:
# =========================================================================
# SOLAR GEOMETRY  (from canopy_lib.py: Calcbeta, calcEccentricity)
# =========================================================================

def solar_elevation(lat_deg, day_of_year, hour_utc):
    """
    Solar elevation angle β (degrees).
    
    From GEE-MEGAN canopy_lib.Calcbeta:
        sinβ = A + B × cos(2π × (hour − 12) / 24)
    where A, B depend on latitude and solar declination.
    
    Parameters
    ----------
    lat_deg     : np.ndarray – Latitude (degrees N), shape (ny, nx)
    day_of_year : int
    hour_utc    : float      – Hour in UTC (0–23, fractional ok)
    
    Returns
    -------
    beta_deg : np.ndarray – Solar elevation (degrees); negative = below horizon
    """
    lat_rad = np.radians(lat_deg)
    
    # Solar declination (radians)
    dec = -23.4 * np.cos(2 * np.pi * (day_of_year + 10) / 365.0)
    dec_rad = np.radians(dec)
    
    sinA = np.sin(lat_rad) * np.sin(dec_rad)
    cosA = np.cos(lat_rad) * np.cos(dec_rad)
    
    # Hour angle
    hour_angle = 2 * np.pi * (hour_utc - 12.0) / 24.0
    
    sin_beta = sinA + cosA * np.cos(hour_angle)
    beta_deg = np.degrees(np.arcsin(np.clip(sin_beta, -1, 1)))
    
    return beta_deg


def eccentricity_factor(day_of_year):
    """
    Earth-Sun distance correction factor.
    From canopy_lib.calcEccentricity.
    
    E = 1 + 0.033 × cos(2π × (day − 10) / 365)
    """
    return 1.0 + 0.033 * np.cos(2 * np.pi * (day_of_year - 10) / 365.0)


def solar_fractions(PPFD, sin_beta, day_of_year):
    """
    Partition total PPFD into beam/diffuse for visible & NIR.
    
    Based on GEE-MEGAN canopy_lib.solarFractions.
    Uses the ratio of actual to potential max solar to estimate
    cloud/diffuse fraction.
    
    Parameters
    ----------
    PPFD       : np.ndarray – Total PPFD (µmol m⁻² s⁻¹)
    sin_beta   : np.ndarray – sin(solar elevation)
    day_of_year : int
    
    Returns
    -------
    dict with keys: 'Qbeamv', 'Qdiffv', 'Qbeamn', 'Qdiffn'
        All in W m⁻² (visible/NIR)
    """
    SOLAR_CONSTANT = 1367.0  # W m⁻²
    
    Ecc = eccentricity_factor(day_of_year)
    
    # Total solar as W/m² (from PPFD: PPFD ≈ Solar × 2.25)
    Solar = PPFD / 2.25
    
    # Maximum possible solar (clear sky, top of atmosphere projected)
    Maxsolar = np.maximum(sin_beta * SOLAR_CONSTANT * Ecc, 0.001)
    
    # Ratio of actual to max
    Ratio = np.clip(Solar / Maxsolar, 0.0, 0.9)
    
    # Fraction of PAR in total solar
    FracPAR = np.where(Ratio >= 0.7, 0.45,
              np.where(Ratio >= 0.3, 0.45 + 0.18 * (0.7 - Ratio) / 0.4,
              0.55 + 0.32 * (0.3 - Ratio) / 0.3))
    
    # PAR (visible) and NIR
    SolarV = Solar * FracPAR          # Visible (PAR) in W/m²
    SolarN = Solar * (1.0 - FracPAR)  # NIR in W/m²
    
    # Diffuse fraction of visible
    RatioV = np.clip(SolarV / (Maxsolar * 0.5), 0, 0.99)
    FracDiffV = np.where(RatioV >= 0.7, 0.2,
                np.where(RatioV >= 0.2, 0.2 + 0.5 * (0.7 - RatioV) / 0.5,
                0.7 + 0.3 * (0.2 - RatioV) / 0.2))
    
    Qdiffv = np.maximum(SolarV * FracDiffV, 0.0)
    Qbeamv = np.maximum(SolarV - Qdiffv, 0.0)
    
    # Diffuse fraction of NIR (similar to visible but with different coefficients)
    RatioN = np.clip(SolarN / (Maxsolar * 0.5), 0, 0.99)
    FracDiffN = np.where(RatioN >= 0.7, 0.2,
                np.where(RatioN >= 0.2, 0.2 + 0.5 * (0.7 - RatioN) / 0.5,
                0.7 + 0.3 * (0.2 - RatioN) / 0.2))
    
    Qdiffn = np.maximum(SolarN * FracDiffN, 0.0)
    Qbeamn = np.maximum(SolarN - Qdiffn, 0.0)
    
    return {
        'Qbeamv': Qbeamv,  # Beam visible (W/m²)
        'Qdiffv': Qdiffv,  # Diffuse visible (W/m²)
        'Qbeamn': Qbeamn,  # Beam NIR (W/m²)
        'Qdiffn': Qdiffn,  # Diffuse NIR (W/m²)
    }


def canopy_ppfd_sunshade(LAI, sin_beta, Qbeamv, Qdiffv, n_layers=5):
    """
    Compute sunlit and shaded PPFD at each canopy layer using
    Gaussian integration (GEE-MEGAN canopy_lib approach).
    
    Uses Beer's law extinction for beam and diffuse radiation.
    
    Parameters
    ----------
    LAI      : np.ndarray – Total LAI
    sin_beta : np.ndarray – sin(solar elevation angle)
    Qbeamv   : np.ndarray – Beam visible radiation (W m⁻²)
    Qdiffv   : np.ndarray – Diffuse visible radiation (W m⁻²)
    n_layers : int        – Number of canopy layers
    
    Returns
    -------
    list of dicts, one per layer:
        {'SunPPFD': ..., 'ShadePPFD': ..., 'Sunfrac': ..., 'weight': ...}
    """
    # Gaussian quadrature points and weights for 5 layers
    if n_layers == 5:
        dist  = [0.0469101, 0.2307534, 0.5000000, 0.7692465, 0.9530899]
        wt    = [0.1184635, 0.2393144, 0.2844444, 0.2393144, 0.1184635]
    elif n_layers == 3:
        dist  = [0.1127016, 0.5000000, 0.8872984]
        wt    = [0.2777778, 0.4444444, 0.2777778]
    else:
        dist  = [0.5]
        wt    = [1.0]
    
    # Extinction coefficients
    scat = 0.2  # leaf scattering coefficient for PAR
    
    # Beam extinction (depends on solar angle)
    Kb = np.where(sin_beta > 0.01, 0.5 / np.maximum(sin_beta, 0.01), 50.0)
    Kd = 0.78  # diffuse extinction coefficient
    
    layers = []
    for i in range(n_layers):
        # LAI depth at this Gaussian point
        LAI_depth = LAI * dist[i]
        
        # Sunlit fraction at this depth (Beer's law)
        Sunfrac = np.exp(-Kb * LAI_depth)
        Sunfrac = np.clip(Sunfrac, 0, 1)
        
        # Diffuse PAR penetrating to this depth
        Qdiff_layer = Qdiffv * np.exp(-Kd * LAI_depth) * (1 - scat)
        
        # Beam PAR at this depth
        Qbeam_layer = Qbeamv * np.exp(-Kb * LAI_depth) * (1 - scat)
        
        # Sunlit leaf PPFD (receives both direct beam + diffuse)
        # Convert W/m² to µmol/m²/s (PAR): × 4.57
        SunPPFD = (Qbeam_layer / np.maximum(sin_beta, 0.01) * sin_beta + Qdiff_layer) * 4.57
        SunPPFD = np.maximum(SunPPFD, 0.0)
        
        # Shaded leaf PPFD (only diffuse)
        ShadePPFD = Qdiff_layer * 4.57
        ShadePPFD = np.maximum(ShadePPFD, 0.0)
        
        # Specific leaf weight (proportional to light availability)
        SLW = np.exp(-Kb * LAI_depth)
        
        layers.append({
            'SunPPFD':   SunPPFD,
            'ShadePPFD': ShadePPFD,
            'Sunfrac':   Sunfrac,
            'weight':    wt[i],
            'SLW':       SLW,
        })
    
    return layers


print("Solar geometry & canopy radiation functions defined.")


## 6. WRF Meteorology Reader

Reads WRF output files (`wrfout_d02_*`) and extracts the variables needed
by MEGAN:

| WRF Variable | Description | Units |
|-------------|-------------|-------|
| `T2` | 2-m temperature | K |
| `SWDOWN` | Downward shortwave at surface | W m⁻² |
| `U10`, `V10` | 10-m wind components | m s⁻¹ |
| `Q2` | 2-m water vapor mixing ratio | kg kg⁻¹ |
| `PSFC` | Surface pressure | Pa |
| `XLAT`, `XLONG` | Grid coordinates | degrees |

**PPFD conversion**: `PPFD (µmol m⁻² s⁻¹) = SWDOWN (W m⁻²) × 2.25`
(following GEE-MEGAN canopy_lib.py, line 690).


In [ ]:
# =========================================================================
# WRF SOURCE PREPARATION (Option A + Option B)
# =========================================================================

def _parse_wrf_datetime_from_text(text):
    import re
    m = re.search(r'(\d{4}-\d{2}-\d{2})[_:](\d{2})[_:](\d{2})[_:](\d{2})', text)
    if not m:
        return None
    return datetime.strptime(f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}", "%Y-%m-%d %H:%M:%S")


def prepare_wrf_inputs(
    source_mode, wrf_dir, link_list_file, cache_dir,
    start_date, end_date, warmup_hours=240
):
    """
    Prepare WRF files for simulation.

    source_mode='local': uses wrf_dir directly (recommended for low-memory local machines).
    source_mode='links': downloads only required files into cache_dir.
    """
    if source_mode == 'local':
        return wrf_dir

    if source_mode != 'links':
        raise ValueError("WRF_SOURCE_MODE must be 'local' or 'links'")

    if not os.path.exists(link_list_file):
        raise FileNotFoundError(f"WRF link list not found: {link_list_file}")

    import urllib.request
    import zipfile

    os.makedirs(cache_dir, exist_ok=True)

    t0 = start_date - timedelta(hours=warmup_hours)
    t1 = end_date + timedelta(hours=23)

    with open(link_list_file, 'r', encoding='utf-8') as f:
        links = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith('#')]

    selected = []
    for link in links:
        dt = _parse_wrf_datetime_from_text(link)
        if dt is None:
            continue
        if t0 <= dt <= t1:
            selected.append((dt, link))

    selected.sort(key=lambda x: x[0])
    print(f"Preparing {len(selected)} WRF files from links...")

    for dt, link in selected:
        base_name = f"wrfout_d02_{dt:%Y-%m-%d_%H_00_00}"
        out_nc = os.path.join(cache_dir, base_name)
        if os.path.exists(out_nc):
            continue

        tmp_target = os.path.join(cache_dir, f"{base_name}.download")
        try:
            urllib.request.urlretrieve(link, tmp_target)
            if zipfile.is_zipfile(tmp_target):
                with zipfile.ZipFile(tmp_target, 'r') as zf:
                    members = [m for m in zf.namelist() if 'wrfout_d02_' in os.path.basename(m)]
                    if not members:
                        raise RuntimeError(f"No wrfout file found inside ZIP for {base_name}")
                    member = members[0]
                    with zf.open(member) as src, open(out_nc, 'wb') as dst:
                        dst.write(src.read())
                os.remove(tmp_target)
            else:
                os.replace(tmp_target, out_nc)
        except Exception as exc:
            print(f"Warning: failed to download {link}: {exc}")
            if os.path.exists(tmp_target):
                os.remove(tmp_target)

    return cache_dir

# =========================================================================
# WRF DATA READER
# =========================================================================

def find_wrf_files(wrf_dir, prefix, start_dt, end_dt):
    """
    Find all WRF output files in wrf_dir matching the date range
    [start_dt, end_dt] (inclusive).
    
    Expected filename pattern: {prefix}YYYY-MM-DD_HH_00_00
    
    Parameters
    ----------
    wrf_dir  : str      – Directory containing wrfout files
    prefix   : str      – e.g. 'wrfout_d02_'
    start_dt : datetime – Start (inclusive)
    end_dt   : datetime – End (inclusive, up to 23:00)
    
    Returns
    -------
    list of (datetime, filepath) sorted by time.
    """
    import re
    pattern = os.path.join(wrf_dir, f"{prefix}*")
    all_files = sorted(glob.glob(pattern))
    
    result = []
    for fp in all_files:
        basename = os.path.basename(fp)
        # Try to parse date from filename
        # Pattern: wrfout_d02_2020-01-01_00_00_00
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})_(\d{2})_(\d{2})', basename)
        if m:
            dt_str = f"{m.group(1)} {m.group(2)}:{m.group(3)}:{m.group(4)}"
            dt = datetime.strptime(dt_str, "%Y-%m-%d %H:%M:%S")
            if start_dt <= dt <= end_dt:
                result.append((dt, fp))
    
    result.sort(key=lambda x: x[0])
    return result


def read_wrf_timestep(filepath, time_idx=0):
    """
    Read a single WRF output file and extract MEGAN-relevant variables.
    
    Parameters
    ----------
    filepath : str
    time_idx : int – Time index within the file (default 0)
    
    Returns
    -------
    dict with keys: 'T2', 'SWDOWN', 'PPFD', 'U10', 'V10', 'WSPD',
                    'Q2', 'PSFC', 'XLAT', 'XLONG', 'datetime'
    All arrays have shape (ny, nx).
    """
    ds = xr.open_dataset(filepath)
    
    # Extract time
    times = ds['Times'].values if 'Times' in ds else None
    
    # Get variables (handle both Time and time dimension names)
    def get_var(name, alt_name=None):
        if name in ds:
            v = ds[name]
        elif alt_name and alt_name in ds:
            v = ds[alt_name]
        else:
            return None
        # Select time
        for dim in ['Time', 'time']:
            if dim in v.dims:
                v = v.isel({dim: time_idx})
        return v.values.astype(np.float64)
    
    T2     = get_var('T2')               # 2m temperature (K)
    SWDOWN = get_var('SWDOWN')           # Shortwave down (W/m²)
    U10    = get_var('U10')              # 10m U-wind (m/s)
    V10    = get_var('V10')              # 10m V-wind (m/s)
    Q2     = get_var('Q2')               # 2m mixing ratio (kg/kg)
    PSFC   = get_var('PSFC')             # Surface pressure (Pa)
    XLAT   = get_var('XLAT', 'XLAT_M')   # Latitude
    XLONG  = get_var('XLONG', 'XLONG_M') # Longitude
    
    ds.close()
    
    # Derived quantities
    PPFD = SWDOWN * 2.25 if SWDOWN is not None else None  # µmol/m²/s
    WSPD = np.sqrt(U10**2 + V10**2) if U10 is not None and V10 is not None else None
    
    return {
        'T2': T2,
        'SWDOWN': SWDOWN,
        'PPFD': PPFD,
        'U10': U10,
        'V10': V10,
        'WSPD': WSPD,
        'Q2': Q2,
        'PSFC': PSFC,
        'XLAT': XLAT,
        'XLONG': XLONG,
    }


def compute_grid_areas(XLAT, XLONG):
    """
    Compute grid cell areas (m²) from lat/lon using the Haversine
    approximation on the grid spacing.
    
    Parameters
    ----------
    XLAT, XLONG : np.ndarray, shape (ny, nx) – Grid coordinates (degrees)
    
    Returns
    -------
    np.ndarray : Area per grid cell (m²), shape (ny, nx)
    """
    R = 6371000.0  # Earth radius (m)
    ny, nx = XLAT.shape
    
    # Approximate dx and dy from coordinate differences
    # dy (m): latitude spacing
    dlat = np.zeros_like(XLAT)
    dlat[1:-1, :] = (XLAT[2:, :] - XLAT[:-2, :]) / 2.0
    dlat[0, :]  = XLAT[1, :] - XLAT[0, :]
    dlat[-1, :] = XLAT[-1, :] - XLAT[-2, :]
    dy = np.abs(dlat) * np.pi / 180.0 * R
    
    # dx (m): longitude spacing (corrected for latitude)
    dlon = np.zeros_like(XLONG)
    dlon[:, 1:-1] = (XLONG[:, 2:] - XLONG[:, :-2]) / 2.0
    dlon[:, 0]  = XLONG[:, 1] - XLONG[:, 0]
    dlon[:, -1] = XLONG[:, -1] - XLONG[:, -2]
    dx = np.abs(dlon) * np.pi / 180.0 * R * np.cos(np.radians(XLAT))
    
    return dx * dy




def infer_domain_bbox(user_bbox=None, wrf_dir=None, wrf_prefix='wrfout_d02_'):
    """Return (south, west, north, east) for satellite data extraction."""
    if user_bbox is not None:
        if len(user_bbox) != 4:
            raise ValueError('SATELLITE_BBOX must have 4 values: (south, west, north, east).')
        south, west, north, east = [float(v) for v in user_bbox]
        return south, west, north, east

    if wrf_dir:
        candidates = sorted(glob.glob(os.path.join(wrf_dir, f"{wrf_prefix}*")))
        if candidates:
            met0 = read_wrf_timestep(candidates[0])
            xlat = np.asarray(met0['XLAT'])
            xlon = np.asarray(met0['XLONG'])
            return float(np.nanmin(xlat)), float(np.nanmin(xlon)), float(np.nanmax(xlat)), float(np.nanmax(xlon))

    raise RuntimeError('Could not infer SATELLITE_BBOX. Provide SATELLITE_BBOX or make at least one WRF file available.')


def load_wrf_grid_geometry(wrf_dir, wrf_prefix='wrfout_d02_'):
    """Load WRF grid geometry for optional ERA5 regridding."""
    candidates = sorted(glob.glob(os.path.join(wrf_dir, f"{wrf_prefix}*"))) if wrf_dir else []
    if not candidates:
        return None
    met0 = read_wrf_timestep(candidates[0])
    return {'XLAT': np.asarray(met0['XLAT']), 'XLONG': np.asarray(met0['XLONG'])}


def fetch_era5_from_codeocean(bbox, time_range, cache_dir):
    """
    Attempt to fetch ERA5 from CodeOcean capsule 4836770.

    Expected environment variables for automated download:
      - CODEOCEAN_ERA5_FILE_URL (direct file URL)
      - optional CODEOCEAN_TOKEN (******
    """
    import requests

    os.makedirs(cache_dir, exist_ok=True)
    target_file = os.path.join(cache_dir, 'era5_codeocean_raw.nc')
    if os.path.exists(target_file):
        return [target_file]

    capsule_url = os.environ.get('CODEOCEAN_CAPSULE_URL', 'https://codeocean.com/capsule/4836770/tree/v1')
    file_url = os.environ.get('CODEOCEAN_ERA5_FILE_URL')
    token = os.environ.get('CODEOCEAN_TOKEN')

    try:
        requests.get(capsule_url, timeout=20)
    except Exception as exc:
        raise RuntimeError(f'CodeOcean capsule not reachable: {exc}') from exc

    if not file_url:
        raise RuntimeError('CODEOCEAN_ERA5_FILE_URL not configured for direct ERA5 download.')

    headers = {'Authorization': f'******'} if token else {}
    resp = requests.get(file_url, headers=headers, stream=True, timeout=300)
    if resp.status_code >= 400:
        raise RuntimeError(f'CodeOcean ERA5 download failed with HTTP {resp.status_code}.')

    with open(target_file, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    return [target_file]


def fetch_era5_from_cds(bbox, time_range, cache_dir):
    """Fetch ERA5 single-level fields from Copernicus CDS with daily caching."""
    try:
        import cdsapi
    except Exception as exc:
        raise RuntimeError('cdsapi is required for CDS fallback. Install cdsapi and configure ~/.cdsapirc or CDSAPI_KEY.') from exc

    os.makedirs(cache_dir, exist_ok=True)
    client = cdsapi.Client(quiet=True)
    start_dt, end_dt = time_range
    south, west, north, east = bbox

    retrieved_files = []
    day = start_dt.date()
    while day <= end_dt.date():
        out_fp = os.path.join(cache_dir, f'era5_{day:%Y%m%d}.nc')
        if os.path.exists(out_fp):
            retrieved_files.append(out_fp)
            day += timedelta(days=1)
            continue

        request = {
            'product_type': 'reanalysis',
            'variable': [
                '2m_temperature',
                'surface_solar_radiation_downwards',
                '10m_u_component_of_wind',
                '10m_v_component_of_wind',
                'surface_pressure',
                '2m_dewpoint_temperature',
            ],
            'year': f'{day.year:04d}',
            'month': f'{day.month:02d}',
            'day': f'{day.day:02d}',
            'time': [f'{h:02d}:00' for h in range(24)],
            'area': [north, west, south, east],
            'format': 'netcdf',
        }

        client.retrieve('reanalysis-era5-single-levels', request, out_fp)
        retrieved_files.append(out_fp)
        day += timedelta(days=1)

    return retrieved_files


def _first_available_dataarray(ds, candidates):
    for name in candidates:
        if name in ds:
            return ds[name]
    return None


def _compute_q2_from_dewpoint(td_k, psfc_pa):
    """Compute near-surface specific humidity proxy from dewpoint and pressure."""
    e = 611.2 * np.exp((17.67 * (td_k - 273.15)) / (td_k - 29.65))
    q = 0.622 * e / np.maximum(psfc_pa - 0.378 * e, 1.0)
    return np.clip(q, 0.0, 0.1)


def _standardize_era5_dataset(ds):
    """Normalize ERA5/CodeOcean fields to WRF-like variable names and dims."""
    t2_da = _first_available_dataarray(ds, ['T2', 't2m', '2m_temperature'])
    sw_da = _first_available_dataarray(ds, ['SWDOWN', 'ssrd', 'surface_solar_radiation_downwards'])
    u10_da = _first_available_dataarray(ds, ['U10', 'u10', '10m_u_component_of_wind'])
    v10_da = _first_available_dataarray(ds, ['V10', 'v10', '10m_v_component_of_wind'])
    psfc_da = _first_available_dataarray(ds, ['PSFC', 'sp', 'surface_pressure'])
    q2_da = _first_available_dataarray(ds, ['Q2', 'q', '2m_specific_humidity', 'specific_humidity'])

    if t2_da is None or sw_da is None or u10_da is None or v10_da is None or psfc_da is None:
        raise RuntimeError('ERA5 dataset is missing one or more required fields: T2/SWDOWN/U10/V10/PSFC.')

    if sw_da.name and ('ssrd' in sw_da.name.lower() or 'surface_solar_radiation_downwards' in sw_da.name.lower()):
        sw_da = sw_da / 3600.0

    if q2_da is None:
        td_da = _first_available_dataarray(ds, ['d2m', '2m_dewpoint_temperature'])
        if td_da is None:
            raise RuntimeError('ERA5 dataset missing Q2 and dewpoint (cannot derive specific humidity).')
        q2_da = xr.DataArray(
            _compute_q2_from_dewpoint(td_da.values, psfc_da.values),
            dims=td_da.dims,
            coords=td_da.coords,
            name='Q2',
        )

    rename_dims = {}
    if 'valid_time' in t2_da.dims:
        rename_dims['valid_time'] = 'time'
    if rename_dims:
        t2_da = t2_da.rename(rename_dims)
        sw_da = sw_da.rename(rename_dims)
        u10_da = u10_da.rename(rename_dims)
        v10_da = v10_da.rename(rename_dims)
        psfc_da = psfc_da.rename(rename_dims)
        q2_da = q2_da.rename(rename_dims)

    lat_da = _first_available_dataarray(ds, ['XLAT', 'lat', 'latitude'])
    lon_da = _first_available_dataarray(ds, ['XLONG', 'lon', 'longitude'])
    if lat_da is None or lon_da is None:
        if 'latitude' in ds.coords and 'longitude' in ds.coords:
            lat_vals = ds['latitude'].values
            lon_vals = ds['longitude'].values
            lons2d, lats2d = np.meshgrid(lon_vals, lat_vals)
        else:
            raise RuntimeError('Could not locate latitude/longitude coordinates in ERA5 dataset.')
    else:
        lat_vals = lat_da.values
        lon_vals = lon_da.values
        if lat_vals.ndim == 1 and lon_vals.ndim == 1:
            lons2d, lats2d = np.meshgrid(lon_vals, lat_vals)
        else:
            lats2d = lat_vals
            lons2d = lon_vals

    sample = t2_da
    spatial_dims = [d for d in sample.dims if d != 'time']
    if len(spatial_dims) != 2:
        raise RuntimeError(f'Expected 2 spatial dimensions in ERA5 fields; got {sample.dims}')
    y_dim, x_dim = spatial_dims

    rename_spatial = {}
    if y_dim != 'y':
        rename_spatial[y_dim] = 'y'
    if x_dim != 'x':
        rename_spatial[x_dim] = 'x'

    def _ren(da):
        return da.rename(rename_spatial) if rename_spatial else da

    t2_da = _ren(t2_da)
    sw_da = _ren(sw_da)
    u10_da = _ren(u10_da)
    v10_da = _ren(v10_da)
    psfc_da = _ren(psfc_da)
    q2_da = _ren(q2_da)

    ds_out = xr.Dataset(
        data_vars={
            'T2': t2_da.astype(np.float64),
            'SWDOWN': sw_da.astype(np.float64),
            'U10': u10_da.astype(np.float64),
            'V10': v10_da.astype(np.float64),
            'Q2': q2_da.astype(np.float64),
            'PSFC': psfc_da.astype(np.float64),
            'XLAT': (('y', 'x'), np.asarray(lats2d, dtype=np.float64)),
            'XLONG': (('y', 'x'), np.asarray(lons2d, dtype=np.float64)),
        },
    )

    if 'time' in t2_da.coords:
        ds_out = ds_out.assign_coords(time=t2_da['time'].values)
    elif 'time' in ds.coords:
        ds_out = ds_out.assign_coords(time=ds['time'].values)

    return ds_out


def regrid_era5_to_wrf(era5_data, wrf_grid):
    """Regrid ERA5 data to a WRF curvilinear grid using scipy.griddata."""
    if wrf_grid is None:
        return era5_data

    from scipy.interpolate import griddata

    src_lat = np.asarray(era5_data['XLAT'].values)
    src_lon = np.asarray(era5_data['XLONG'].values)
    tgt_lat = np.asarray(wrf_grid['XLAT'])
    tgt_lon = np.asarray(wrf_grid['XLONG'])

    src_points = np.column_stack([src_lat.ravel(), src_lon.ravel()])
    tgt_points = (tgt_lat, tgt_lon)

    regridded = {
        'XLAT': (('y', 'x'), tgt_lat),
        'XLONG': (('y', 'x'), tgt_lon),
    }

    for var_name in ['T2', 'SWDOWN', 'U10', 'V10', 'Q2', 'PSFC']:
        da = era5_data[var_name]
        if 'time' in da.dims:
            slices = []
            for t in range(da.sizes['time']):
                values = np.asarray(da.isel(time=t).values).ravel()
                arr = griddata(src_points, values, tgt_points, method='linear')
                if np.isnan(arr).any():
                    arr_nn = griddata(src_points, values, tgt_points, method='nearest')
                    arr = np.where(np.isnan(arr), arr_nn, arr)
                slices.append(arr)
            regridded[var_name] = (('time', 'y', 'x'), np.asarray(slices, dtype=np.float64))
        else:
            values = np.asarray(da.values).ravel()
            arr = griddata(src_points, values, tgt_points, method='linear')
            if np.isnan(arr).any():
                arr_nn = griddata(src_points, values, tgt_points, method='nearest')
                arr = np.where(np.isnan(arr), arr_nn, arr)
            regridded[var_name] = (('y', 'x'), np.asarray(arr, dtype=np.float64))

    coords = {}
    if 'time' in era5_data.coords:
        coords['time'] = era5_data['time'].values

    return xr.Dataset(regridded, coords=coords)


def _write_standard_satellite_daily_files(ds_std, cache_dir):
    """Write standardized daily satellite files as satmet_YYYYMMDD.nc."""
    if 'time' not in ds_std.coords:
        raise RuntimeError('Satellite dataset must include a time coordinate.')

    times = pd.to_datetime(ds_std['time'].values)
    unique_days = sorted(pd.DatetimeIndex(times).normalize().unique())
    out_files = []

    for day in unique_days:
        day_mask = pd.DatetimeIndex(times).normalize() == day
        if not np.any(day_mask):
            continue
        ds_day = ds_std.isel(time=np.where(day_mask)[0])
        out_fp = os.path.join(cache_dir, f"satmet_{pd.Timestamp(day):%Y%m%d}.nc")
        ds_day.to_netcdf(out_fp)
        out_files.append(out_fp)

    return out_files


def find_satellite_files(sat_dir, start_dt, end_dt):
    """Discover standardized satellite files and map each hourly timestep."""
    sat_files = sorted(glob.glob(os.path.join(sat_dir, 'satmet_*.nc')))
    result = []

    for fp in sat_files:
        with xr.open_dataset(fp) as ds:
            if 'time' not in ds:
                continue
            times = pd.to_datetime(ds['time'].values)
        for idx, dt in enumerate(times):
            py_dt = pd.Timestamp(dt).to_pydatetime().replace(tzinfo=None)
            if start_dt <= py_dt <= end_dt:
                result.append((py_dt, fp, idx))

    result.sort(key=lambda x: x[0])
    return result


def prepare_satellite_inputs(
    start_date, end_date, satellite_cache_dir,
    bbox=None, wrf_dir=None, wrf_prefix='wrfout_d02_', warmup_hours=240
):
    """Prepare satellite/ERA5 meteorology cache (CodeOcean first, CDS fallback)."""
    os.makedirs(satellite_cache_dir, exist_ok=True)

    t0 = start_date - timedelta(hours=warmup_hours)
    t1 = end_date + timedelta(hours=23)

    existing = find_satellite_files(satellite_cache_dir, t0, t1)
    if existing:
        print(f"Using cached satellite meteorology in: {satellite_cache_dir}")
        return satellite_cache_dir

    domain_bbox = infer_domain_bbox(user_bbox=bbox, wrf_dir=wrf_dir, wrf_prefix=wrf_prefix)
    wrf_grid = load_wrf_grid_geometry(wrf_dir, wrf_prefix=wrf_prefix)
    print(f"Satellite domain bbox (S,W,N,E): {domain_bbox}")

    raw_files = []
    try:
        print('Attempting ERA5 fetch from CodeOcean capsule 4836770...')
        raw_files = fetch_era5_from_codeocean(domain_bbox, (t0, t1), satellite_cache_dir)
        print('CodeOcean ERA5 fetch succeeded.')
    except Exception as codeocean_exc:
        print(f"CodeOcean ERA5 fetch unavailable: {codeocean_exc}")
        print('Falling back to Copernicus CDS...')
        try:
            raw_files = fetch_era5_from_cds(domain_bbox, (t0, t1), satellite_cache_dir)
            print('CDS ERA5 fetch succeeded.')
        except Exception as cds_exc:
            raise RuntimeError(
                f"Satellite meteorology could not be prepared. CodeOcean error: {codeocean_exc}; CDS error: {cds_exc}"
            ) from cds_exc

    standardized_files = []
    for raw_fp in raw_files:
        with xr.open_dataset(raw_fp) as ds_raw:
            ds_std = _standardize_era5_dataset(ds_raw)
        if wrf_grid is not None:
            ds_std = regrid_era5_to_wrf(ds_std, wrf_grid)
        standardized_files.extend(_write_standard_satellite_daily_files(ds_std, satellite_cache_dir))

    if not standardized_files:
        raise RuntimeError('Satellite meteorology preparation finished with no satmet_*.nc outputs.')

    return satellite_cache_dir


def read_satellite_timestep(filepath, time_idx=0):
    """Read one ERA5/satellite timestep into the same dict structure as WRF reader."""
    ds = xr.open_dataset(filepath)

    def get_var(name):
        if name not in ds:
            return None
        da = ds[name]
        if 'time' in da.dims:
            da = da.isel(time=time_idx)
        return da.values.astype(np.float64)

    T2 = get_var('T2')
    SWDOWN = get_var('SWDOWN')
    U10 = get_var('U10')
    V10 = get_var('V10')
    Q2 = get_var('Q2')
    PSFC = get_var('PSFC')
    XLAT = get_var('XLAT')
    XLONG = get_var('XLONG')
    ds.close()

    PPFD = SWDOWN * 2.25 if SWDOWN is not None else None
    WSPD = np.sqrt(U10**2 + V10**2) if U10 is not None and V10 is not None else None

    return {
        'T2': T2,
        'SWDOWN': SWDOWN,
        'PPFD': PPFD,
        'U10': U10,
        'V10': V10,
        'WSPD': WSPD,
        'Q2': Q2,
        'PSFC': PSFC,
        'XLAT': XLAT,
        'XLONG': XLONG,
    }


def prepare_meteorological_inputs(
    meteorological_source,
    wrf_source_mode,
    wrf_dir,
    link_list_file,
    wrf_cache_dir,
    satellite_cache_dir,
    start_date,
    end_date,
    warmup_hours=240,
    satellite_bbox=None,
    wrf_prefix='wrfout_d02_',
):
    """Branching gateway: return directory for chosen meteorological source."""
    source = meteorological_source.lower().strip()
    if source == 'wrf':
        print('Meteorological source selected: WRF')
        return prepare_wrf_inputs(
            source_mode=wrf_source_mode,
            wrf_dir=wrf_dir,
            link_list_file=link_list_file,
            cache_dir=wrf_cache_dir,
            start_date=start_date,
            end_date=end_date,
            warmup_hours=warmup_hours,
        )
    if source == 'satellite':
        print('Meteorological source selected: Satellite/ERA5 (CodeOcean/CDS)')
        return prepare_satellite_inputs(
            start_date=start_date,
            end_date=end_date,
            satellite_cache_dir=satellite_cache_dir,
            bbox=satellite_bbox,
            wrf_dir=wrf_dir,
            wrf_prefix=wrf_prefix,
            warmup_hours=warmup_hours,
        )
    raise ValueError("METEOROLOGICAL_SOURCE must be 'wrf' or 'satellite'.")


def find_meteorology_files(meteorological_source, met_dir, wrf_prefix, start_dt, end_dt):
    """Return list of (datetime, filepath, time_idx) for selected source."""
    source = meteorological_source.lower().strip()
    if source == 'wrf':
        return [(dt, fp, 0) for dt, fp in find_wrf_files(met_dir, wrf_prefix, start_dt, end_dt)]
    if source == 'satellite':
        return find_satellite_files(met_dir, start_dt, end_dt)
    raise ValueError("meteorological_source must be 'wrf' or 'satellite'.")


def read_meteorology_timestep(meteorological_source, filepath, time_idx=0):
    """Dispatch reader while preserving the WRF-like return dictionary."""
    source = meteorological_source.lower().strip()
    if source == 'wrf':
        return read_wrf_timestep(filepath, time_idx=time_idx)
    if source == 'satellite':
        return read_satellite_timestep(filepath, time_idx=time_idx)
    raise ValueError("meteorological_source must be 'wrf' or 'satellite'.")


print('Meteorology reader functions defined (WRF + Satellite/ERA5).')


## 7. LAI and Growth-Form Fraction Reader

Reads the preprocessed input files produced by your companion scripts:

1. **`MEGAN_LAI_CLIM_d02_MEGAN32.nc`** — Monthly LAI climatology on the WRF d02 grid
   (from `build_megan_lai_from_tifs.py`). Dimensions: `(month=12, south_north, west_east)`.

2. **`CT3.SC_Brazil.csv`** — Growth-form fractions from MapBiomas
   (from `build_sc_brazil_csvs.py`). Columns: `CID, ICELL, JCELL, NEEDL, TROPI, BROAD, SHRUB, HERB, CROP`.
   Values in percent (0–100).

The growth-form fractions are used to weight the PFT-specific emission factors.


In [ ]:
# =========================================================================
# LAI AND GROWTH-FORM FRACTION READER
# =========================================================================

def load_lai_climatology(lai_file):
    """
    Load monthly LAI climatology NetCDF.
    
    Parameters
    ----------
    lai_file : str – Path to MEGAN_LAI_CLIM_d02_MEGAN32.nc
    
    Returns
    -------
    np.ndarray : LAI, shape (12, ny, nx), units m² m⁻²
    """
    ds = xr.open_dataset(lai_file)
    lai = ds['LAI'].values.astype(np.float64)  # (12, ny, nx)
    ds.close()
    
    # Clamp to valid range
    lai = np.clip(lai, 0, 10)
    
    print(f"LAI climatology loaded: shape {lai.shape}, "
          f"range [{lai.min():.2f}, {lai.max():.2f}] m² m⁻²")
    return lai


def load_growth_form_fractions(ct3_file, ny, nx):
    """
    Load growth-form fractions from CT3 CSV and reshape to 2D grid.
    
    Parameters
    ----------
    ct3_file : str  – Path to CT3.SC_Brazil.csv
    ny, nx   : int  – Grid dimensions (from WRF)
    
    Returns
    -------
    dict : {growth_form_name: np.ndarray of shape (ny, nx)}
           Values as fractions (0–1).
    """
    df = pd.read_csv(ct3_file)
    
    expected_n = ny * nx
    if len(df) != expected_n:
        raise ValueError(
            f"CT3 has {len(df)} rows but WRF grid has {ny}×{nx}={expected_n} cells. "
            f"Check grid dimensions."
        )
    
    growth_forms = {}
    for gf in ['NEEDL', 'TROPI', 'BROAD', 'SHRUB', 'HERB', 'CROP']:
        if gf in df.columns:
            vals = df[gf].values.astype(np.float64) / 100.0  # percent → fraction
            growth_forms[gf] = vals.reshape(ny, nx)
        else:
            growth_forms[gf] = np.zeros((ny, nx))
    
    # Total PFT fraction per cell
    total = sum(growth_forms.values())
    
    print(f"Growth-form fractions loaded: {ny}×{nx} grid")
    for gf, arr in growth_forms.items():
        print(f"  {gf}: mean={arr.mean():.3f}, max={arr.max():.3f}")
    print(f"  Total cover: mean={total.mean():.3f}")
    
    return growth_forms


def compute_weighted_ef(growth_forms, species_list):
    """
    Compute area-weighted emission factors for each species based on
    growth-form fractions and the PFT→EF mapping.
    
    EF_weighted(i,j,spc) = Σ_gf [ fraction_gf(i,j) × EF(spc, PFT_gf) ]
    
    Normalized by total fraction:
    EF_norm = EF_weighted / max(total_fraction, 0.01)
    
    Parameters
    ----------
    growth_forms : dict {gf_name: np.ndarray (ny, nx)}
    species_list : list of str
    
    Returns
    -------
    dict : {spc_name: np.ndarray (ny, nx) of weighted EF in µg m⁻² h⁻¹}
    dict : total_frac np.ndarray (ny, nx)
    """
    ny, nx = list(growth_forms.values())[0].shape
    
    total_frac = np.zeros((ny, nx))
    for gf, frac in growth_forms.items():
        total_frac += frac
    total_frac_safe = np.maximum(total_frac, 0.01)
    
    ef_weighted = {}
    for spc in species_list:
        ef_map = np.zeros((ny, nx))
        for gf, frac in growth_forms.items():
            pft_idx = GROWTHFORM_PFT_INDEX.get(gf, 15)  # default: bare/other
            ef_val = COMPOUND_EF[spc][pft_idx]
            ef_map += frac * ef_val
        ef_weighted[spc] = ef_map / total_frac_safe
    
    return ef_weighted, total_frac


print("LAI and growth-form reader functions defined.")


## 8. MEGAN Emission Calculation Engine

This is the core computation that combines all gamma factors to produce
hourly emission rates for each species.

### The MEGAN2.1 Emission Equation

For each grid cell and each species $s$:

$$
ER_s = EF_s \times \gamma_{age,s} \times \gamma_{SM} \times \rho_s
    \times \gamma_{LAI} \times \left[
        LDF_s \times \gamma_{LD,s}
        + (1 - LDF_s) \times \gamma_{LI,s}
    \right]
$$

where:
- $EF_s$ = PFT-weighted standard emission factor (µg m⁻² h⁻¹)
- $\gamma_{age,s}$ = leaf age activity factor
- $\gamma_{SM}$ = soil moisture factor (= 1.0 in this version)
- $\rho_s$ = canopy loss/production factor (= 1.0)
- $\gamma_{LAI}$ = LAI scaling factor
- $LDF_s$ = light-dependence fraction (0–1)
- $\gamma_{LI,s} = \exp[\beta_s (T - 303.15)]$ = light-**independent** temperature factor
  (Ealti99). It does **not** depend on light, so stored-pool emissions (e.g. NO,
  low-LDF monoterpenes) correctly persist at night.
- $\gamma_{LD,s}$ = canopy-averaged light-**dependent** activity, integrating the
  sunlit/shade leaf temperature×light response over the canopy with Gaussian
  quadrature (weights $w_l$) and the sunlit fraction $f_{sun,l}$ at each layer $l$:

$$
\gamma_{LD,s} =
    \sum_{l=1}^{L} w_l
    \left[
        f_{sun,l}\,\gamma_{T,s}\,\gamma_{P,sun,l}
        + (1-f_{sun,l})\,\gamma_{T,s}\,\gamma_{P,shade,l}
    \right]
$$

This is the standard MEGAN2.1 LDF split (Guenther et al. 2012, Eq. 9), equivalent
to the `ER_calculate` assembly in GEE-MEGAN `EMPORC.py` (lines 785–787). The light
factor $\gamma_P$ (Ea1p99) is normalized so the daytime canopy-average activity is
$\mathcal{O}(1)$; $\gamma_{LAI}$ supplies the leaf-area scaling.


In [ ]:
# =========================================================================
# EMISSION CALCULATION ENGINE
# =========================================================================

def compute_hourly_emissions(
    T2, PPFD, T24, T240, PPFD24, PPFD240,
    XLAT, XLONG,
    LAI_current, LAI_previous, T_daily_mean,
    growth_forms, ef_weighted, total_frac,
    species_list, day_of_year, hour_utc,
    n_layers=5
):
    """
    Compute MEGAN hourly emission rates for all species.
    
    Parameters
    ----------
    T2          : np.ndarray (ny, nx) – 2m temperature (K)
    PPFD        : np.ndarray (ny, nx) – Instantaneous PPFD (µmol/m²/s)
    T24         : np.ndarray (ny, nx) – 24h mean temperature (K)
    T240        : np.ndarray (ny, nx) – 240h mean temperature (K)
    PPFD24      : np.ndarray (ny, nx) – 24h mean PPFD
    PPFD240     : np.ndarray (ny, nx) – 240h mean PPFD
    XLAT, XLONG : np.ndarray (ny, nx) – Grid coordinates
    LAI_current : np.ndarray (ny, nx) – Current month LAI
    LAI_previous: np.ndarray (ny, nx) – Previous month LAI
    T_daily_mean: np.ndarray (ny, nx) – Daily mean T (K) for leaf age
    growth_forms: dict
    ef_weighted : dict {spc: EF array}
    total_frac  : np.ndarray (ny, nx)
    species_list: list of str
    day_of_year : int
    hour_utc    : float
    n_layers    : int
    
    Returns
    -------
    dict : {spc_name: np.ndarray (ny, nx)} – Emission rate (µg m⁻² h⁻¹)
    """
    ny, nx = T2.shape
    
    # 1. Solar geometry
    beta_deg = solar_elevation(XLAT, day_of_year, hour_utc)
    sin_beta = np.sin(np.radians(np.maximum(beta_deg, 0.0)))
    is_day   = beta_deg > 0.0
    
    # 2. Radiation partitioning
    sfrac = solar_fractions(PPFD, sin_beta, day_of_year)
    
    # 3. Canopy radiation (sunlit/shade PPFD per layer)
    canopy_layers = canopy_ppfd_sunshade(
        LAI_current, sin_beta, sfrac['Qbeamv'], sfrac['Qdiffv'], n_layers
    )
    
    # 4. γ_LAI
    gam_lai = gamma_lai(LAI_current)
    
    # 5. γ_age (per species)
    gam_age_dict = gamma_age(LAI_current, LAI_previous, T_daily_mean, TSTLEN=30)
    
    # 6. Light-INDEPENDENT temperature factor (species-specific, at air temp).
    #    Single canopy-level value (Guenther 2012, Eq. 12); does NOT depend on
    #    light, so it correctly persists at night for stored-pool emitters
    #    (e.g. NO, monoterpenes with low LDF).
    #    (Computed inside the per-species loop because β is species-specific.)
    
    # 7. Per-species emission assembly
    emissions = {}
    
    for spc in species_list:
        # --- Light-DEPENDENT canopy activity (dimensionless, ~O(1)) ---
        # Integrate the sunlit/shade leaf light×temperature response over the
        # canopy using Gaussian quadrature weights and the sunlit fraction at
        # each layer. This is the canopy-average activity per unit leaf area.
        gamma_LD = np.zeros((ny, nx))
        for layer in canopy_layers:
            w   = layer['weight']
            sfr = layer['Sunfrac']
            
            # Leaf temperature ≈ air temperature (PCEEA simplification)
            gT_sun   = gamma_T_lightdep(T2, T24, T240, spc)
            gT_shade = gamma_T_lightdep(T2, T24, T240, spc)
            
            # Light response for sunlit (high PPFD) and shaded (diffuse only) leaves
            gP_sun   = gamma_P_lightdep(layer['SunPPFD'],   PPFD24,        PPFD240,        Pstd=200.0)
            gP_shade = gamma_P_lightdep(layer['ShadePPFD'], PPFD24 * 0.16, PPFD240 * 0.16, Pstd=50.0)
            
            layer_act = sfr * gT_sun * gP_sun + (1.0 - sfr) * gT_shade * gP_shade
            gamma_LD += w * layer_act
        
        # --- Light-INDEPENDENT temperature activity (single value) ---
        gamma_LI = gamma_T_lightindep(T2, spc)
        
        # --- LDF combination (Guenther et al. 2012, Eq. 9) ---
        # gamma_activity = gamma_LAI × [ LDF × gamma_LD + (1 - LDF) × gamma_LI ]
        ldf = LDF[spc]
        gamma_activity = gam_lai * (ldf * gamma_LD + (1.0 - ldf) * gamma_LI)
        
        # --- Other activity factors ---
        gam_a  = gam_age_dict[spc]   # leaf age
        gam_sm = 1.0                 # soil moisture (not modeled; = 1.0)
        rho    = 1.0                 # canopy loss/production (= 1.0)
        
        # --- Final emission rate (µg m⁻² h⁻¹) ---
        EF = ef_weighted[spc]
        ER = EF * gam_a * gam_sm * rho * gamma_activity
        
        emissions[spc] = np.maximum(ER, 0.0)
    
    return emissions


print("Emission calculation engine defined.")


## 9. Main Simulation Loop

This cell runs the MEGAN model over the user-defined time period.

### Processing strategy
- **Day-by-day** processing to manage memory
- **Rolling buffers** for T24/T240/PPFD24/PPFD240 running averages
- Writes one NetCDF output file per simulation day
- Accumulates daily and annual totals

### Running averages
- **T24**: 24-hour running mean of temperature
- **T240**: 240-hour (10-day) running mean of temperature
- **PPFD24**: 24-hour running mean of PPFD
- **PPFD240**: 240-hour running mean of PPFD

These require a **warmup period** of at least 240 hours (~10 days) of
WRF data before the simulation start date.


In [ ]:
# =========================================================================
# MAIN SIMULATION LOOP
# =========================================================================

def run_megan_simulation(
    start_date, end_date,
    wrf_dir, wrf_prefix,
    meteorological_source='wrf',
    lai_clim,
    growth_forms, ef_weighted, total_frac,
    species_list, output_dir,
    n_layers=5, warmup_hours=240
):
    """
    Run MEGAN biogenic emission simulation.
    
    Parameters
    ----------
    start_date, end_date : datetime
    wrf_dir, wrf_prefix  : str
    lai_clim             : np.ndarray (12, ny, nx)
    growth_forms         : dict
    ef_weighted          : dict {spc: EF array}
    total_frac           : np.ndarray
    species_list         : list of str
    output_dir           : str
    n_layers             : int
    warmup_hours         : int
    
    Returns
    -------
    dict : summary statistics
    """
    import sys
    
    ny, nx = lai_clim.shape[1], lai_clim.shape[2]
    
    # Determine full time range (including warmup)
    warmup_start = start_date - timedelta(hours=warmup_hours)
    sim_end = end_date + timedelta(hours=23)
    
    # Find all meteorological files according to selected source
    all_wrf = find_meteorology_files(meteorological_source, wrf_dir, wrf_prefix, warmup_start, sim_end)
    
    if len(all_wrf) == 0:
        print("ERROR: No meteorological files found in the specified date range!")
        print(f"  Looked in: {wrf_dir}")
        print(f"  Date range: {warmup_start} → {sim_end}")
        print(f"  Pattern: {wrf_prefix}*")
        return None
    
    print(f"Found {len(all_wrf)} meteorological timesteps ({meteorological_source})")
    print(f"  First: {all_wrf[0][0]}")
    print(f"  Last:  {all_wrf[-1][0]}")
    
    # Read grid info from first file
    first_met = read_meteorology_timestep(meteorological_source, all_wrf[0][1], time_idx=all_wrf[0][2])
    XLAT  = first_met['XLAT']
    XLONG = first_met['XLONG']
    grid_area = compute_grid_areas(XLAT, XLONG)  # m²
    
    print(f"Grid: {ny} × {nx}, area range: {grid_area.min()/1e6:.2f}–{grid_area.max()/1e6:.2f} km²")
    
    # Rolling average buffers
    T_buffer    = []   # list of T2 arrays (last 240 entries)
    PPFD_buffer = []   # list of PPFD arrays (last 240 entries)
    
    # Output accumulators
    daily_emissions = {}  # {spc: np.ndarray (ny, nx)} in µg/m²
    annual_emissions = {spc: np.zeros((ny, nx)) for spc in species_list}
    n_sim_days = 0
    
    # Track simulation dates
    current_sim_day = None
    hours_in_day = 0
    
    for i, (dt, filepath, time_idx) in enumerate(all_wrf):
        # Read meteorology
        try:
            met = read_meteorology_timestep(meteorological_source, filepath, time_idx=time_idx)
        except Exception as e:
            print(f"  WARNING: Could not read {filepath}: {e}")
            continue
        
        T2   = met['T2']
        PPFD = met['PPFD']
        
        if T2 is None or PPFD is None:
            continue
        
        # Update rolling buffers
        T_buffer.append(T2)
        PPFD_buffer.append(PPFD)
        
        # Keep only last 240 entries
        if len(T_buffer) > 240:
            T_buffer = T_buffer[-240:]
        if len(PPFD_buffer) > 240:
            PPFD_buffer = PPFD_buffer[-240:]
        
        # Compute running averages
        n_t = len(T_buffer)
        T24  = np.mean(T_buffer[-min(24, n_t):], axis=0) if n_t >= 1 else T2
        T240 = np.mean(T_buffer, axis=0)
        PPFD24  = np.mean(PPFD_buffer[-min(24, n_t):], axis=0) if n_t >= 1 else PPFD
        PPFD240 = np.mean(PPFD_buffer, axis=0)
        
        # Skip warmup period
        if dt < start_date:
            if (i + 1) % 48 == 0:
                print(f"  Warmup: {dt:%Y-%m-%d %H:%M} ({n_t}/{warmup_hours} hours buffered)")
            continue
        
        # Day tracking
        sim_day = dt.date()
        if sim_day != current_sim_day:
            # Write previous day's output (if any)
            if current_sim_day is not None and hours_in_day > 0:
                write_daily_output(
                    current_sim_day, daily_emissions, hours_in_day,
                    grid_area, XLAT, XLONG, species_list, output_dir
                )
                n_sim_days += 1
                for spc in species_list:
                    annual_emissions[spc] += daily_emissions[spc]
            
            # Reset daily accumulators
            current_sim_day = sim_day
            daily_emissions = {spc: np.zeros((ny, nx)) for spc in species_list}
            hours_in_day = 0
            
            if n_sim_days % 30 == 0 or n_sim_days == 0:
                print(f"  Processing: {sim_day} (day {n_sim_days + 1})")
        
        # LAI for current and previous month
        month = dt.month
        lai_current  = lai_clim[month - 1]          # 0-indexed
        lai_previous = lai_clim[(month - 2) % 12]   # previous month, wrapping
        
        # Daily mean T (approximate from T24)
        T_daily = T24
        
        # Compute hourly emissions
        doy = dt.timetuple().tm_yday
        
        em = compute_hourly_emissions(
            T2=T2, PPFD=PPFD, T24=T24, T240=T240,
            PPFD24=PPFD24, PPFD240=PPFD240,
            XLAT=XLAT, XLONG=XLONG,
            LAI_current=lai_current, LAI_previous=lai_previous,
            T_daily_mean=T_daily,
            growth_forms=growth_forms, ef_weighted=ef_weighted,
            total_frac=total_frac,
            species_list=species_list,
            day_of_year=doy, hour_utc=dt.hour,
            n_layers=n_layers
        )
        
        # Accumulate (µg/m²/h → µg/m² by summing hourly values)
        for spc in species_list:
            daily_emissions[spc] += em[spc]
        hours_in_day += 1
    
    # Write final day
    if current_sim_day is not None and hours_in_day > 0:
        write_daily_output(
            current_sim_day, daily_emissions, hours_in_day,
            grid_area, XLAT, XLONG, species_list, output_dir
        )
        n_sim_days += 1
        for spc in species_list:
            annual_emissions[spc] += daily_emissions[spc]
    
    # Write annual summary
    write_annual_output(
        start_date, end_date, annual_emissions, n_sim_days,
        grid_area, XLAT, XLONG, species_list, output_dir
    )
    
    print(f"\n{'='*60}")
    print(f"Simulation complete: {n_sim_days} days processed")
    print(f"Output directory: {output_dir}")
    
    return {
        'n_days': n_sim_days,
        'grid_area': grid_area,
        'annual_emissions': annual_emissions,
        'XLAT': XLAT,
        'XLONG': XLONG,
    }

print("Simulation loop defined.")


## 10. NetCDF Output Writers

Output files are **CF-1.8 compliant** with proper CRS metadata for direct
loading in **QGIS** (and other GIS tools).

### Output files
| File | Content | Units |
|------|---------|-------|
| `MEGAN_emissions_YYYY-MM-DD.nc` | Daily per-species emission rates | µg m⁻² h⁻¹ (hourly mean) and tonnes day⁻¹ |
| `MEGAN_emissions_annual_YYYY.nc` | Annual per-species totals | tonnes year⁻¹ (for the simulated period) |


In [ ]:
# =========================================================================
# NETCDF OUTPUT WRITERS (CF-1.8 compliant, QGIS-ready)
# =========================================================================

def derive_reporting_species(emissions_dict):
    """Map internal MEGAN species to requested reporting groups."""
    template = next(iter(emissions_dict.values()))
    zeros = np.zeros_like(template)

    def _sum(keys):
        out = np.zeros_like(template)
        for k in keys:
            out += emissions_dict.get(k, zeros)
        return out

    derived = {
        'ISOP': emissions_dict.get('ISOP', zeros),
        'MTRY': _sum(['MYRC', 'SABI', 'LIMO', 'A_3CAR', 'OCIM', 'BPIN', 'APIN', 'OMTP']),
        'SESQ': _sum(['FARN', 'BCAR', 'OSQT']),
        'CH3OH': emissions_dict.get('MEOH', zeros),
        # HCHO and CH3COOH are not explicit primary emitted compounds in this simplified setup.
        # They are reported as conservative direct-emission placeholders.
        'HCHO': zeros.copy(),
        'CH3COOH': zeros.copy(),
        'OTHER_VOC': _sum(['MBO', 'ACTO', 'CO', 'NO', 'BIDER', 'STRESS', 'OTHER']),
    }
    return derived


def _add_crs(ds):
    ds['crs'] = xr.DataArray(
        np.int32(0),
        attrs={
            'grid_mapping_name': 'latitude_longitude',
            'semi_major_axis': 6378137.0,
            'inverse_flattening': 298.257223563,
            'epsg_code': 'EPSG:4326',
            'spatial_ref': 'EPSG:4326',
            'crs_wkt': (
                'GEOGCS["WGS 84",DATUM["WGS_1984",'
                'SPHEROID["WGS 84",6378137,298.257223563]],'
                'PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],'
                'AUTHORITY["EPSG","4326"]]'
            ),
        }
    )


def write_daily_output(
    sim_day, daily_emissions, hours_in_day,
    grid_area, XLAT, XLONG, species_list, output_dir
):
    ny, nx = XLAT.shape
    ds = xr.Dataset()

    ds.coords['south_north'] = np.arange(ny)
    ds.coords['west_east']   = np.arange(nx)
    ds['latitude']  = (('south_north', 'west_east'), XLAT.astype(np.float32))
    ds['longitude'] = (('south_north', 'west_east'), XLONG.astype(np.float32))
    ds['grid_area'] = (('south_north', 'west_east'), grid_area.astype(np.float32))

    ds['latitude'].attrs  = {'units': 'degrees_north', 'standard_name': 'latitude'}
    ds['longitude'].attrs = {'units': 'degrees_east',  'standard_name': 'longitude'}
    ds['grid_area'].attrs = {'units': 'm2', 'long_name': 'Grid cell area'}

    reporting = derive_reporting_species(daily_emissions)
    out_species = [s for s in REPORT_SPECIES if s in reporting]

    for spc in out_species:
        rate = reporting[spc] / max(hours_in_day, 1)
        ds[f'{spc}_rate'] = (('south_north', 'west_east'), rate.astype(np.float32))
        ds[f'{spc}_rate'].attrs = {
            'units': 'ug m-2 h-1',
            'long_name': f'{spc} biogenic emission rate (daily mean)',
            'coordinates': 'latitude longitude',
            'grid_mapping': 'crs',
        }

        tonnes_day = reporting[spc] * grid_area / 1e12
        ds[f'{spc}_tonnes_per_day'] = (('south_north', 'west_east'), tonnes_day.astype(np.float32))
        ds[f'{spc}_tonnes_per_day'].attrs = {
            'units': 'tonnes day-1',
            'long_name': f'{spc} biogenic emission (tonnes per day per grid cell)',
            'coordinates': 'latitude longitude',
            'grid_mapping': 'crs',
        }

    ds.attrs = {
        'Conventions': 'CF-1.8',
        'title': f'MEGAN biogenic emissions – {sim_day}',
        'source': 'MEGAN 3.2 project setup with Guenther et al. 2006/2012 gamma factors',
        'references': 'Guenther et al. 2006 ACP; Guenther et al. 2012 GMD; Hoinaski et al. 2024 ESSD',
        'history': f'Created {datetime.now():%Y-%m-%d %H:%M:%S}',
        'simulation_date': str(sim_day),
        'hours_simulated': hours_in_day,
        'institution': 'Generated by MEGAN Biogenic Emissions Notebook',
        'crs': 'EPSG:4326',
    }

    _add_crs(ds)

    outpath = os.path.join(output_dir, f"MEGAN_emissions_{sim_day}.nc")
    encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for var in ds.data_vars if var != 'crs'}
    ds.to_netcdf(outpath, format='NETCDF4', encoding=encoding)
    ds.close()


def write_annual_output(
    start_date, end_date, annual_emissions, n_days,
    grid_area, XLAT, XLONG, species_list, output_dir
):
    ny, nx = XLAT.shape
    ds = xr.Dataset()

    ds.coords['south_north'] = np.arange(ny)
    ds.coords['west_east']   = np.arange(nx)
    ds['latitude']  = (('south_north', 'west_east'), XLAT.astype(np.float32))
    ds['longitude'] = (('south_north', 'west_east'), XLONG.astype(np.float32))
    ds['grid_area'] = (('south_north', 'west_east'), grid_area.astype(np.float32))

    ds['latitude'].attrs  = {'units': 'degrees_north', 'standard_name': 'latitude'}
    ds['longitude'].attrs = {'units': 'degrees_east',  'standard_name': 'longitude'}
    ds['grid_area'].attrs = {'units': 'm2', 'long_name': 'Grid cell area'}

    total_hours = n_days * 24
    reporting = derive_reporting_species(annual_emissions)
    out_species = [s for s in REPORT_SPECIES if s in reporting]

    for spc in out_species:
        tonnes = reporting[spc] * grid_area / 1e12
        ds[f'{spc}_tonnes_total'] = (('south_north', 'west_east'), tonnes.astype(np.float32))
        ds[f'{spc}_tonnes_total'].attrs = {
            'units': 'tonnes year-1',
            'long_name': f'{spc} total biogenic emission over simulation period',
            'coordinates': 'latitude longitude',
            'grid_mapping': 'crs',
        }

        rate_mean = reporting[spc] / max(total_hours, 1)
        ds[f'{spc}_rate_mean'] = (('south_north', 'west_east'), rate_mean.astype(np.float32))
        ds[f'{spc}_rate_mean'].attrs = {
            'units': 'ug m-2 h-1',
            'long_name': f'{spc} mean biogenic emission rate over simulation period',
            'coordinates': 'latitude longitude',
            'grid_mapping': 'crs',
        }

    ds.attrs = {
        'Conventions': 'CF-1.8',
        'title': f'MEGAN biogenic emissions — period total ({start_date:%Y-%m-%d} to {end_date:%Y-%m-%d})',
        'source': 'MEGAN 3.2 project setup with Guenther et al. 2006/2012 gamma factors',
        'references': 'Guenther et al. 2006 ACP; Guenther et al. 2012 GMD; Hoinaski et al. 2024 ESSD',
        'history': f'Created {datetime.now():%Y-%m-%d %H:%M:%S}',
        'simulation_start': str(start_date.date()),
        'simulation_end': str(end_date.date()),
        'n_days_simulated': n_days,
        'total_hours': total_hours,
        'institution': 'Generated by MEGAN Biogenic Emissions Notebook',
        'crs': 'EPSG:4326',
    }

    _add_crs(ds)

    outpath = os.path.join(output_dir, f"MEGAN_emissions_annual_{start_date:%Y%m%d}_{end_date:%Y%m%d}.nc")
    encoding = {var: {'zlib': True, 'complevel': 4, 'dtype': 'float32'} for var in ds.data_vars if var != 'crs'}
    ds.to_netcdf(outpath, format='NETCDF4', encoding=encoding)
    ds.close()

    print(f"
Annual output: {outpath}")
    print(f"Period: {start_date:%Y-%m-%d} → {end_date:%Y-%m-%d} ({n_days} days)")
    print(f"
Domain-total emissions (tonnes):")
    print(f"{'Species':<10} {'Total (tonnes)':>15} {'Mean rate (µg/m²/h)':>20}")
    print('-' * 50)
    for spc in out_species:
        total_t = (reporting[spc] * grid_area / 1e12).sum()
        mean_r  = reporting[spc].mean() / max(total_hours, 1)
        print(f"{spc:<10} {total_t:>15.2f} {mean_r:>20.4f}")


print('NetCDF output writers defined.')


## 11. Run the Simulation

Execute the cell below to run the MEGAN model for the configured time period.

**Before running:**
1. Ensure WRF files are in `input/WRF/` (see README.md)
2. Ensure LAI climatology is in `input/LAI/`
3. Ensure CT3 growth-form CSV is in `input/LULC/`


In [ ]:
# =========================================================================
# LOAD INPUTS
# =========================================================================

# Load LAI
lai_clim = load_lai_climatology(LAI_FILE)
ny, nx = lai_clim.shape[1], lai_clim.shape[2]

# Load growth-form fractions
growth_forms = load_growth_form_fractions(CT3_FILE, ny, nx)

# Compute weighted emission factors
ef_weighted, total_frac = compute_weighted_ef(growth_forms, SPECIES_LIST)

print("\nWeighted emission factors (domain mean, µg m⁻² h⁻¹):")
for spc in SPECIES_LIST[:5]:
    print(f"  {spc}: {ef_weighted[spc].mean():.1f}")
print("  ...")


In [ ]:
# =========================================================================
# RUN SIMULATION
# =========================================================================

met_processing_dir = prepare_meteorological_inputs(
    meteorological_source=METEOROLOGICAL_SOURCE,
    wrf_source_mode=WRF_SOURCE_MODE,
    wrf_dir=WRF_DIR,
    link_list_file=WRF_LINK_LIST_FILE,
    wrf_cache_dir=WRF_CACHE_DIR,
    satellite_cache_dir=SATELLITE_CACHE_DIR,
    start_date=START_DATE,
    end_date=END_DATE,
    warmup_hours=WARMUP_HOURS,
    satellite_bbox=SATELLITE_BBOX,
    wrf_prefix=WRF_PREFIX,
)

results = run_megan_simulation(
    start_date   = START_DATE,
    end_date     = END_DATE,
    wrf_dir      = met_processing_dir,
    wrf_prefix   = WRF_PREFIX,
    meteorological_source = METEOROLOGICAL_SOURCE,
    lai_clim     = lai_clim,
    growth_forms = growth_forms,
    ef_weighted  = ef_weighted,
    total_frac   = total_frac,
    species_list = SPECIES_LIST,
    output_dir   = OUTPUT_DIR,
    n_layers     = N_CANOPY_LAYERS,
    warmup_hours = WARMUP_HOURS,
)


## 12. Diagnostics and Quick Visualization

Optional cell to inspect results and produce summary plots.
Requires `matplotlib` (`pip install matplotlib`).


In [ ]:
# =========================================================================
# DIAGNOSTICS
# =========================================================================

if results is not None:
    try:
        import matplotlib
        matplotlib.use('Agg')
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        key_species = ['ISOP', 'APIN', 'LIMO', 'MEOH', 'NO', 'BCAR']
        
        for ax, spc in zip(axes.ravel(), key_species):
            if spc in results['annual_emissions']:
                data = results['annual_emissions'][spc] * results['grid_area'] / 1e12
                im = ax.pcolormesh(
                    results['XLONG'], results['XLAT'], data,
                    cmap='YlOrRd', shading='auto'
                )
                ax.set_title(f'{spc} (tonnes)', fontsize=12)
                ax.set_xlabel('Longitude')
                ax.set_ylabel('Latitude')
                plt.colorbar(im, ax=ax, shrink=0.8)
        
        fig.suptitle(
            f'MEGAN Biogenic Emissions — {START_DATE:%Y-%m-%d} to {END_DATE:%Y-%m-%d}',
            fontsize=14, fontweight='bold'
        )
        plt.tight_layout()
        
        fig_path = os.path.join(OUTPUT_DIR, 'emissions_summary.png')
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Summary figure saved: {fig_path}")
        
    except ImportError:
        print("matplotlib not installed — skipping visualization.")
        print("Install with: pip install matplotlib")
else:
    print("No results to visualize. Check that WRF files are available.")


## 13. Validation Against BRAIN MEGAN Output (Optional)

If you have BRAIN MEGAN output files (from SciDB dataset
[10.57760/sciencedb.14561](https://doi.org/10.57760/sciencedb.14561)),
you can compare them with this notebook's output.

Edit the cell below to point to a BRAIN MEGAN file.


In [ ]:
# =========================================================================
# OPTIONAL VALIDATION AGAINST BRAIN
# =========================================================================

# Path to a BRAIN MEGAN output file (e.g., MEGAN/megan_2020001.nc)
BRAIN_MEGAN_FILE = None  # Set to file path, e.g.: os.path.join(BASE_DIR, 'validation', 'megan_2020001.nc')

if BRAIN_MEGAN_FILE and os.path.exists(BRAIN_MEGAN_FILE):
    brain_ds = xr.open_dataset(BRAIN_MEGAN_FILE)
    print("BRAIN MEGAN file loaded:")
    print(f"  Variables: {list(brain_ds.data_vars)}")
    print(f"  Dimensions: {dict(brain_ds.dims)}")
    print(f"  Time range: {brain_ds.coords.get('TSTEP', 'N/A')}")
    
    # Example comparison for isoprene
    # (Variable names in BRAIN output may differ — adjust as needed)
    for var in brain_ds.data_vars:
        print(f"  {var}: min={float(brain_ds[var].min()):.4f}, "
              f"max={float(brain_ds[var].max()):.4f}, "
              f"mean={float(brain_ds[var].mean()):.4f}")
    brain_ds.close()
else:
    print("No BRAIN validation file specified. Skipping validation.")
    print("To validate, set BRAIN_MEGAN_FILE above to the path of a BRAIN MEGAN output .nc file.")


## 14. Loading Outputs in QGIS

### Step-by-step

1. **Open QGIS** and create a new project.

2. **Add the NetCDF layer**:
   - Go to *Layer → Add Layer → Add Raster Layer*
   - Browse to `output/MEGAN_emissions_annual_*.nc`
   - QGIS will list the available sub-datasets (one per species variable)
   - Select e.g. `ISOP_tonnes_total` and click *Add*

3. **Verify CRS**: The files use **EPSG:4326** (WGS 84). QGIS should
   recognize this from the embedded CRS metadata.

4. **Style the layer**:
   - Right-click the layer → *Properties → Symbology*
   - Choose *Singleband pseudocolor*
   - Select a color ramp (e.g., *YlOrRd* for emissions)
   - Classify and apply

5. **Add basemap** (optional):
   - Install the *QuickMapServices* plugin
   - Go to *Web → QuickMapServices → OSM → OSM Standard*

### Tips
- Use `ISOP_tonnes_total` or `ISOP_rate_mean` for annual maps
- Use daily files (`MEGAN_emissions_YYYY-MM-DD.nc`) for time-series analysis
- The `latitude` and `longitude` 2D coordinate variables enable correct
  georeferencing even though the grid is curvilinear (WRF Lambert conformal)

---

## 15. Folder/File Structure

```
MEGAN_Biogenic_Emissions/
├── notebook/
│   └── MEGAN_biogenic_emissions.ipynb   ← THIS NOTEBOOK
├── input/
│   ├── WRF/                             ← Place wrfout_d02_* files here
│   │   └── wrfout_d02_2020-01-01_00_00_00
│   │   └── wrfout_d02_2020-01-01_01_00_00
│   │   └── ...
│   ├── LAI/
│   │   └── MEGAN_LAI_CLIM_d02_MEGAN32.nc   ← From build_megan_lai_from_tifs.py
│   └── LULC/
│       └── CT3.SC_Brazil.csv                ← From build_sc_brazil_csvs.py
├── output/                              ← Generated .nc files appear here
│   ├── MEGAN_emissions_2020-01-01.nc
│   ├── MEGAN_emissions_2020-01-02.nc
│   ├── ...
│   ├── MEGAN_emissions_annual_20200101_20201231.nc
│   └── emissions_summary.png
├── scripts/                             ← Reference preprocessing scripts
│   ├── build_megan_lai_from_tifs.py
│   ├── build_sc_brazil_csvs.py
│   └── MEGAN_LAI_WRFd02.py             (GEE JavaScript)
├── docs/
│   └── (supplementary documentation)
└── README.md
```


## 15. Reproducibility Statement


In [ ]:
# =========================================================================
# REPRODUCIBILITY
# =========================================================================
import sys
print(f"Python:    {sys.version}")
print(f"NumPy:     {np.__version__}")
print(f"xarray:    {xr.__version__}")
print(f"pandas:    {pd.__version__}")
try:
    import netCDF4
    print(f"netCDF4:   {netCDF4.__version__}")
except ImportError:
    print("netCDF4:   not installed")
try:
    import scipy
    print(f"scipy:     {scipy.__version__}")
except ImportError:
    print("scipy:     not installed")
try:
    import matplotlib
    print(f"matplotlib:{matplotlib.__version__}")
except ImportError:
    print("matplotlib:not installed (optional)")
